# 2026 NLP Competition Solution

Ноутбук выполняет полный цикл: загрузка данных, EDA, обучение, инференс, постпроцессинг и сохранение `sample_submission.csv`.

In [ ]:
import ast
import platform
import random
import re
import warnings
from dataclasses import dataclass
from html import unescape
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from scipy.sparse import hstack
from sklearn.base import clone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.model_selection import KFold
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import OneHotEncoder
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModel, AutoModelForSequenceClassification, AutoTokenizer

try:
    from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
except ImportError:
    MultilabelStratifiedKFold = None

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

SEED = 322
TARGET_COLUMNS = ["label_0", "label_1", "label_2", "label_3", "label_4"]


def detect_data_root() -> Path:
    # Local run first.
    local_root = Path(".")
    if (local_root / "train.csv").exists() and (local_root / "test.csv").exists() and (local_root / "sample_submission.csv").exists():
        return local_root

    # Kaggle competition input (folder name can vary by slug).
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for train_path in kaggle_input.rglob("train.csv"):
            root = train_path.parent
            if (root / "test.csv").exists() and (root / "sample_submission.csv").exists():
                return root

    raise FileNotFoundError("Could not find train.csv, test.csv, sample_submission.csv")


PROJECT_ROOT = detect_data_root()


@dataclass
class DataBundle:
    train: pd.DataFrame
    test: pd.DataFrame
    sample_submission: pd.DataFrame


def set_global_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def parse_target(value: str) -> np.ndarray:
    arr = np.asarray(ast.literal_eval(value), dtype=np.int64)
    if arr.shape != (5,):
        raise ValueError(f"Target must have 5 labels, got {arr.shape}.")
    if not np.isin(arr, [0, 1]).all():
        raise ValueError("Target values must be binary.")
    return arr


def load_competition_data(data_dir: Path | str = ".") -> DataBundle:
    data_dir = Path(data_dir)
    train = pd.read_csv(data_dir / "train.csv", sep="\t")
    test = pd.read_csv(data_dir / "test.csv", sep="\t")
    sample_submission = pd.read_csv(data_dir / "sample_submission.csv")
    return DataBundle(train=train, test=test, sample_submission=sample_submission)


def validate_schema(train: pd.DataFrame, test: pd.DataFrame, sample_submission: pd.DataFrame) -> None:
    if set(train.columns) != {"id", "source", "title", "text", "publication_date", "target"}:
        raise ValueError(f"Unexpected train columns: {train.columns.tolist()}")
    if set(test.columns) != {"id", "source", "title", "text", "publication_date"}:
        raise ValueError(f"Unexpected test columns: {test.columns.tolist()}")
    if set(sample_submission.columns) != {"id", "target"}:
        raise ValueError(f"Unexpected submission columns: {sample_submission.columns.tolist()}")


def add_target_columns(train: pd.DataFrame) -> pd.DataFrame:
    parsed = np.vstack(train["target"].map(parse_target).to_numpy())
    out = train.copy()
    for i, c in enumerate(TARGET_COLUMNS):
        out[c] = parsed[:, i]
    return out


def extract_targets(df: pd.DataFrame) -> np.ndarray:
    return df[TARGET_COLUMNS].to_numpy(dtype=np.int64)


def format_submission_targets(binary_matrix: np.ndarray) -> list[str]:
    return ["[" + ",".join(map(str, row.astype(int).tolist())) + "]" for row in binary_matrix]


def hamming_score(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    # Kaggle score behaves as hamming loss: lower is better.
    return float(np.not_equal(y_true, y_pred).mean())


TAG_RE = re.compile(r"<[^>]+>")
SPACE_RE = re.compile(r"\s+")


def clean_text(value: str) -> str:
    if not isinstance(value, str):
        value = "" if pd.isna(value) else str(value)
    text = unescape(value)
    text = TAG_RE.sub(" ", text)
    text = text.replace("\n", " ").replace("\r", " ").replace("\t", " ")
    return SPACE_RE.sub(" ", text).strip()


def compose_text_features(
    df: pd.DataFrame,
    include_source: bool = True,
    include_date: bool = True,
) -> pd.Series:
    title = df["title"].fillna("").map(clean_text)
    text = df["text"].fillna("").map(clean_text)

    parts = ["[TITLE] " + title, "[TEXT] " + text]

    if include_source:
        source = df["source"].fillna("").astype(str)
        parts.insert(0, "[SRC] " + source)

    if include_date:
        dt = pd.to_datetime(df["publication_date"], errors="coerce")
        year = dt.dt.year.fillna(-1).astype(int).astype(str)
        month = dt.dt.month.fillna(-1).astype(int).astype(str)
        parts.insert(1 if include_source else 0, "[YEAR] " + year + " [MONTH] " + month)

    combined = parts[0]
    for p in parts[1:]:
        combined = combined + " " + p
    return combined


def _mean_pool(last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    pooled = (last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
    return pooled


def build_transformer_embeddings(texts: pd.Series, model_name: str, batch_size: int, max_length: int, cache_path: Path | None = None) -> np.ndarray:
    if cache_path is not None and cache_path.exists():
        return np.load(cache_path)["embeddings"]

    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()

    chunks = []
    with torch.inference_mode():
        for start in tqdm(range(0, len(texts), batch_size), desc=f"Embeddings:{model_name}"):
            batch = [f"passage: {t}" for t in texts.iloc[start:start + batch_size].tolist()]
            encoded = tokenizer(batch, padding=True, truncation=True, max_length=max_length, return_tensors="pt").to(device)
            with torch.autocast(device_type="cuda", enabled=(device == "cuda")):
                outputs = model(**encoded)
                pooled = _mean_pool(outputs.last_hidden_state, encoded["attention_mask"])
                pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)
            chunks.append(pooled.float().cpu().numpy())

    embeddings = np.vstack(chunks)
    if cache_path is not None:
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(cache_path, embeddings=embeddings)
    return embeddings


def build_tfidf_features(train_texts: pd.Series, test_texts: pd.Series, max_features_word: int = 120000, max_features_char: int = 80000):
    word = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.98, sublinear_tf=True, max_features=max_features_word)
    char = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_df=0.98, sublinear_tf=True, max_features=max_features_char)
    x_train = hstack([word.fit_transform(train_texts), char.fit_transform(train_texts)]).tocsr()
    x_test = hstack([word.transform(test_texts), char.transform(test_texts)]).tocsr()
    return x_train, x_test


def build_disentangled_sparse_features(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    max_title_features: int = 60000,
    max_text_word_features: int = 180000,
    max_text_char_features: int = 90000,
):
    title_train = train_df["title"].fillna("").map(clean_text)
    title_test = test_df["title"].fillna("").map(clean_text)
    text_train = train_df["text"].fillna("").map(clean_text)
    text_test = test_df["text"].fillna("").map(clean_text)

    title_vec = TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.98,
        sublinear_tf=True,
        max_features=max_title_features,
    )
    text_word_vec = TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.98,
        sublinear_tf=True,
        max_features=max_text_word_features,
    )
    text_char_vec = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=3,
        max_df=0.98,
        sublinear_tf=True,
        max_features=max_text_char_features,
    )

    x_title_train = title_vec.fit_transform(title_train)
    x_title_test = title_vec.transform(title_test)
    x_text_word_train = text_word_vec.fit_transform(text_train)
    x_text_word_test = text_word_vec.transform(text_test)
    x_text_char_train = text_char_vec.fit_transform(text_train)
    x_text_char_test = text_char_vec.transform(text_test)

    dt_train = pd.to_datetime(train_df["publication_date"], errors="coerce")
    dt_test = pd.to_datetime(test_df["publication_date"], errors="coerce")

    cat_train = pd.DataFrame(
        {
            "source": train_df["source"].fillna("unknown").astype(str),
            "month": dt_train.dt.month.fillna(-1).astype(int).astype(str),
            "year": dt_train.dt.year.fillna(-1).astype(int).astype(str),
        }
    )
    cat_test = pd.DataFrame(
        {
            "source": test_df["source"].fillna("unknown").astype(str),
            "month": dt_test.dt.month.fillna(-1).astype(int).astype(str),
            "year": dt_test.dt.year.fillna(-1).astype(int).astype(str),
        }
    )

    try:
        cat_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        cat_encoder = OneHotEncoder(handle_unknown="ignore", sparse=True)

    x_cat_train = cat_encoder.fit_transform(cat_train)
    x_cat_test = cat_encoder.transform(cat_test)

    x_train = hstack(
        [x_title_train, x_text_word_train, x_text_char_train, x_cat_train],
        format="csr",
    )
    x_test = hstack(
        [x_title_test, x_text_word_test, x_text_char_test, x_cat_test],
        format="csr",
    )

    return x_train, x_test


def build_base_classifier(seed: int = SEED, C: float = 2.0, class_weight: str | None = "balanced"):
    base = LogisticRegression(
        C=C,
        max_iter=600,
        solver="liblinear",
        random_state=seed,
        class_weight=class_weight,
    )
    return OneVsRestClassifier(base)


def build_sgd_classifier(seed: int = SEED, alpha: float = 3e-5, class_weight: str | None = "balanced"):
    base = SGDClassifier(
        loss="log_loss",
        penalty="l2",
        alpha=alpha,
        max_iter=3000,
        tol=1e-3,
        random_state=seed,
        class_weight=class_weight,
    )
    return OneVsRestClassifier(base)


def make_multilabel_splitter(n_splits: int = 5, seed: int = SEED):
    if MultilabelStratifiedKFold is not None:
        return MultilabelStratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return KFold(n_splits=n_splits, shuffle=True, random_state=seed)


def make_time_splits(df: pd.DataFrame, n_splits: int = 5, min_train_folds: int = 1):
    """Forward-chaining time split: train uses only past, valid uses next block."""
    dt = pd.to_datetime(df["publication_date"], errors="coerce")
    dt = dt.fillna(pd.Timestamp("1970-01-01"))
    order = np.argsort(dt.to_numpy())
    fold_bins = np.array_split(order, n_splits)

    splits = []
    for i in range(max(min_train_folds, 1), n_splits):
        va_idx = fold_bins[i]
        tr_idx = np.concatenate(fold_bins[:i])
        if len(tr_idx) == 0 or len(va_idx) == 0:
            continue
        splits.append((tr_idx, va_idx))

    if not splits:
        # Fallback to a single split if data is too small.
        cut = max(1, len(order) // 2)
        splits = [(order[:cut], order[cut:])]

    return splits


def fit_cv_models(x, y: np.ndarray, estimator, splitter, score_fn):
    oof = np.zeros((y.shape[0], y.shape[1]), dtype=np.float32)
    oof_mask = np.zeros(y.shape[0], dtype=bool)
    models, fold_scores = [], []

    if hasattr(splitter, "split"):
        split_iter = splitter.split(x, y)
    else:
        split_iter = splitter

    for fold_idx, (tr_idx, va_idx) in enumerate(split_iter, start=1):
        model = clone(estimator)
        model.fit(x[tr_idx], y[tr_idx])
        proba = model.predict_proba(x[va_idx])
        oof[va_idx] = proba
        oof_mask[va_idx] = True
        pred = (proba >= 0.5).astype(np.int64)
        score = score_fn(y[va_idx], pred)
        fold_scores.append(float(score))
        models.append(model)
        print(f"Fold {fold_idx}: hamming_loss={score:.6f}")
    return {"oof_proba": oof, "oof_mask": oof_mask, "models": models, "fold_scores": fold_scores}


def fit_full_model(x, y: np.ndarray, estimator):
    model = clone(estimator)
    model.fit(x, y)
    return model


def find_best_thresholds(y_true: np.ndarray, y_proba: np.ndarray, score_fn, grid: np.ndarray | None = None):
    if grid is None:
        grid = np.linspace(0.05, 0.95, 91)
    thresholds = np.full(y_true.shape[1], 0.5, dtype=np.float32)
    best_global = score_fn(y_true, (y_proba >= thresholds).astype(np.int64))
    for label in range(y_true.shape[1]):
        best_t, best_score = thresholds[label], best_global
        for t in grid:
            trial = thresholds.copy()
            trial[label] = float(t)
            score = score_fn(y_true, (y_proba >= trial).astype(np.int64))
            if score < best_score:
                best_t, best_score = float(t), score
        thresholds[label] = best_t
        best_global = best_score
    return thresholds, float(best_global)


def apply_thresholds(y_proba: np.ndarray, thresholds: np.ndarray) -> np.ndarray:
    return (y_proba >= thresholds).astype(np.int64)


class TextDataset(Dataset):
    def __init__(self, texts: list[str], targets: np.ndarray | None = None):
        self.texts = texts
        self.targets = targets

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        item = {"text": self.texts[idx]}
        if self.targets is not None:
            item["targets"] = self.targets[idx].astype(np.float32)
        return item


def collate_fn_builder(tokenizer, max_length: int):
    def collate_fn(batch):
        texts = [x["text"] for x in batch]
        encoded = tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )
        if "targets" in batch[0]:
            targets = torch.tensor([x["targets"] for x in batch], dtype=torch.float32)
            encoded["labels"] = targets
        return encoded

    return collate_fn


def predict_proba_model(model, loader, device: str) -> np.ndarray:
    model.eval()
    probs = []
    with torch.no_grad():
        for batch in loader:
            labels = batch.pop("labels", None)
            batch = {k: v.to(device) for k, v in batch.items()}
            logits = model(**batch).logits
            p = torch.sigmoid(logits).cpu().numpy()
            probs.append(p)
    return np.vstack(probs)


def train_one_fold_transformer(
    train_texts: list[str],
    train_targets: np.ndarray,
    val_texts: list[str],
    model_name: str,
    seed: int,
    epochs: int,
    lr: float,
    batch_size: int,
    max_length: int,
) -> tuple[np.ndarray, AutoModelForSequenceClassification, AutoTokenizer]:
    set_global_seed(seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=train_targets.shape[1],
        problem_type="multi_label_classification",
    ).to(device)

    train_ds = TextDataset(train_texts, train_targets)
    val_ds = TextDataset(val_texts, None)

    collate_fn = collate_fn_builder(tokenizer, max_length=max_length)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_ds, batch_size=batch_size * 2, shuffle=False, collate_fn=collate_fn)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    model.train()
    for epoch in range(epochs):
        epoch_losses = []
        for batch in train_loader:
            labels = batch.pop("labels").to(device)
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch, labels=labels)
            loss = outputs.loss
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            epoch_losses.append(loss.item())
        print(f"Epoch {epoch + 1}/{epochs} loss={np.mean(epoch_losses):.6f}")

    val_proba = predict_proba_model(model, val_loader, device=device)
    return val_proba, model, tokenizer


def calibrate_with_logreg(y_true: np.ndarray, y_proba: np.ndarray, y_test_proba: np.ndarray, seed: int = SEED):
    # GO component: classical model over neural probabilities.
    meta = OneVsRestClassifier(
        LogisticRegression(
            C=6.0,
            solver="liblinear",
            max_iter=1200,
            random_state=seed,
        )
    )
    meta.fit(y_proba, y_true)
    calibrated_train = meta.predict_proba(y_proba)
    calibrated_test = meta.predict_proba(y_test_proba)
    return calibrated_train, calibrated_test


set_global_seed(SEED)
print(f"Seed fixed to: {SEED}")
print(f"Python: {platform.python_version()}")

In [ ]:
bundle = load_competition_data(PROJECT_ROOT)
train_df = bundle.train
test_df = bundle.test
sample_submission = bundle.sample_submission

validate_schema(train_df, test_df, sample_submission)
train_df = add_target_columns(train_df)

print("Data root:", PROJECT_ROOT)
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Sample submission shape:", sample_submission.shape)
display(train_df.head(2))

In [ ]:
# EDA
y = extract_targets(train_df)
label_freq = pd.Series(y.mean(axis=0), index=TARGET_COLUMNS, name="positive_ratio")
cardinality = pd.Series(y.sum(axis=1), name="labels_per_sample")

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
label_freq.plot(kind="bar", ax=axes[0], title="Label positive ratio")
cardinality.value_counts().sort_index().plot(kind="bar", ax=axes[1], title="Label cardinality")

tmp = train_df.copy()
tmp["title_len"] = tmp["title"].fillna("").astype(str).str.len()
tmp["text_len"] = tmp["text"].fillna("").astype(str).str.len()
sns.scatterplot(data=tmp.sample(min(4000, len(tmp)), random_state=SEED), x="title_len", y="text_len", s=8, ax=axes[2])
axes[2].set_title("Title vs text length")
plt.tight_layout()
plt.show()

display(label_freq.to_frame())
display(train_df["source"].value_counts().head(20).to_frame("count"))

In [ ]:
# v1.0 feature recipe for neural stack
best_variant_name = "title_text_source_date"

train_texts = compose_text_features(train_df, include_source=True, include_date=True)
test_texts = compose_text_features(test_df, include_source=True, include_date=True)

print("Selected feature recipe:", best_variant_name)
display(train_texts.head(2))

In [ ]:
# Neural training config
if str(PROJECT_ROOT).startswith("/kaggle/input"):
    ARTIFACTS_DIR = Path("/kaggle/working/artifacts")
else:
    ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "cointegrated/rubert-tiny2"
RUN_MODE = "full"  # debug/full

MAX_LENGTH = 256
TRAIN_EPOCHS = 2 if RUN_MODE == "full" else 1
BATCH_SIZE = 24 if RUN_MODE == "full" else 16
LR = 2e-5
N_SPLITS = 5 if RUN_MODE == "full" else 3
ENSEMBLE_SEEDS = [322, 42] if RUN_MODE == "full" else [322]

print("Model:", MODEL_NAME)
print("Run mode:", RUN_MODE)
print("Epochs:", TRAIN_EPOCHS, "Batch:", BATCH_SIZE, "LR:", LR)
print("CV folds:", N_SPLITS, "Seeds:", ENSEMBLE_SEEDS)

In [ ]:
# Build forward time splits once for all neural stages
CV_STRATEGY = "forward_time"
time_splits = make_time_splits(train_df, n_splits=N_SPLITS, min_train_folds=1)

print("CV strategy:", CV_STRATEGY)
print("Forward folds:", len(time_splits))

In [ ]:
# Neural CV training + GO calibration
y = extract_targets(train_df)
train_text_list = train_texts.tolist()
test_text_list = test_texts.tolist()

n_samples = len(train_df)
n_labels = y.shape[1]
seed_oof_list = []
seed_test_list = []
seed_losses = []
cv_mask = np.zeros(n_samples, dtype=bool)

for seed_value in ENSEMBLE_SEEDS:
    print(f"\n=== Seed {seed_value} ===")
    oof_seed = np.zeros((n_samples, n_labels), dtype=np.float32)
    test_fold_probs = []

    for fold_idx, (tr_idx, va_idx) in enumerate(time_splits, start=1):
        print(f"Fold {fold_idx}/{len(time_splits)} train={len(tr_idx)} valid={len(va_idx)}")

        val_proba, model, tokenizer = train_one_fold_transformer(
            train_texts=[train_text_list[i] for i in tr_idx],
            train_targets=y[tr_idx],
            val_texts=[train_text_list[i] for i in va_idx],
            model_name=MODEL_NAME,
            seed=seed_value,
            epochs=TRAIN_EPOCHS,
            lr=LR,
            batch_size=BATCH_SIZE,
            max_length=MAX_LENGTH,
        )

        oof_seed[va_idx] = val_proba
        cv_mask[va_idx] = True

        test_loader = DataLoader(
            TextDataset(test_text_list, None),
            batch_size=BATCH_SIZE * 2,
            shuffle=False,
            collate_fn=collate_fn_builder(tokenizer, max_length=MAX_LENGTH),
        )
        fold_test_proba = predict_proba_model(model, test_loader, device=("cuda" if torch.cuda.is_available() else "cpu"))
        test_fold_probs.append(fold_test_proba)

        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    seed_test_mean = np.mean(test_fold_probs, axis=0)
    seed_oof_list.append(oof_seed)
    seed_test_list.append(seed_test_mean)

    pred_seed = (oof_seed[cv_mask] >= 0.5).astype(np.int64)
    loss_seed = hamming_score(y[cv_mask], pred_seed)
    seed_losses.append(loss_seed)
    print(f"OOF neural seed={seed_value} hamming_loss@0.5 (masked): {loss_seed:.6f}")

cv_oof_neural = np.mean(seed_oof_list, axis=0)
test_proba_neural = np.mean(seed_test_list, axis=0)

neural_pred = (cv_oof_neural[cv_mask] >= 0.5).astype(np.int64)
neural_loss = hamming_score(y[cv_mask], neural_pred)
print(f"OOF neural ensemble hamming_loss@0.5 (masked): {neural_loss:.6f}")

# GO calibration block
cv_oof_calibrated, test_proba_calibrated = calibrate_with_logreg(
    y_true=y[cv_mask],
    y_proba=cv_oof_neural[cv_mask],
    y_test_proba=test_proba_neural,
    seed=SEED,
)

cal_pred = (cv_oof_calibrated >= 0.5).astype(np.int64)
cal_loss = hamming_score(y[cv_mask], cal_pred)
print(f"OOF calibrated (GO) hamming_loss@0.5 (masked): {cal_loss:.6f}")

cv_oof_proba = np.zeros_like(cv_oof_neural)
cv_oof_proba[cv_mask] = cv_oof_calibrated
test_proba_final = test_proba_calibrated

baseline_score = cal_loss
print(f"Selected OOF hamming_loss@0.5: {baseline_score:.6f}")

In [ ]:
# Threshold tuning on selected OOF probabilities (blended if available)
best_thresholds, tuned_score = find_best_thresholds(
    y_true=y[cv_mask],
    y_proba=cv_oof_proba[cv_mask],
    score_fn=hamming_score,
)
print("Best thresholds:", best_thresholds)
print(f"Tuned OOF hamming_loss (masked): {tuned_score:.6f}")

In [ ]:
# Submission from calibrated neural probabilities
test_pred = apply_thresholds(test_proba_final, best_thresholds)

submission = sample_submission.copy()
submission["id"] = test_df["id"].values
submission["target"] = format_submission_targets(test_pred)

if str(PROJECT_ROOT).startswith("/kaggle/input"):
    output_path = Path("/kaggle/working/sample_submission.csv")
else:
    output_path = PROJECT_ROOT / "sample_submission.csv"

submission.to_csv(output_path, index=False)
print("Saved submission:", output_path)
display(submission.head())

In [ ]:
print("=== Reproducibility report ===")
print("Seed:", SEED)
print("Model:", MODEL_NAME)
print("CV strategy:", CV_STRATEGY)
print("Ensemble seeds:", ENSEMBLE_SEEDS)
print("Epochs:", TRAIN_EPOCHS, "Batch:", BATCH_SIZE, "LR:", LR)
print("Neural seed losses@0.5:", [round(v, 6) for v in seed_losses])
print("Selected feature recipe:", best_variant_name)
print("OOF selected loss@0.5:", round(baseline_score, 6))
print("OOF tuned loss:", round(tuned_score, 6))
print("OOF coverage:", int(cv_mask.sum()), "/", len(cv_mask))
print("Train rows:", len(train_df), "| Test rows:", len(test_df))